# Cosine few-shot evaluation

Loads a checkpoint produced by a pretrain experiment and runs prototypical-
network few-shot evaluation. Everything for this run is written to
`experiments/<experiment_name>/evaluations/<eval_name>/`:

- `eval_config.json` — every parameter you set below + the architecture
  auto-loaded from the pretrain experiment (under `arch_from_pretrain`)
- `eval_metadata.json` — timestamp, git SHA, OA/AA/Kappa summary
- `results.json` — full results including per-class accuracy
- `eval.log` — evaluation log
- `plots/` — confusion matrix, per-class accuracy, t-SNE, ...

The `experiment_name` is the key: architecture (`embed_dim`, `num_heads`,
`num_layers`, `patch_size`, `lambda_factor`, `dropout`, projection-head
settings) is read automatically from `experiments/<name>/pretrain_config.yaml`,
so `eval_params` only needs evaluation-shaped knobs.

Multiple evals against the same experiment are normal — just pick a different
`eval_name` each time.

In [ ]:
# === Parameters ===
# Which pretraining experiment to evaluate.
experiment_name = "houston_enhanced_spatial_mask_test_run1_seed52"

# Short name for THIS evaluation run — becomes the eval subdir.
eval_name = "houston_enhanced_spatial_mask_test_run5_seed52"

# Checkpoint epoch to load. None = latest in checkpoints/ (or checkpoint_final.pth if present).
epoch = None

# Alternative: pass an explicit checkpoint path (or the string 'random' for an untrained baseline).
# Overrides `epoch` when set.
checkpoint = "experiments/houston_enhanced_spatial_mask_test_run1_seed52/checkpoints/checkpoint_epoch_1500.pth"

# Which device/GPU to evaluate on. Options:
#   "cuda"    -> first visible GPU (cuda:0); errors if no CUDA.
#   "cuda:N"  -> specific GPU index (e.g. "cuda:1", "cuda:2", "cuda:3").
#   "auto"    -> cuda:0 if available, else cpu.
#   "cpu"     -> force CPU.
# Run `!nvidia-smi` in a cell to see which GPUs are free.
device = "cuda:3"

# Whether to keep the pretraining projection head at evaluation time.
#   False -> discard the projection, use raw encoder features (typical for
#            metric-learning / prototypical-network evaluation).
#   True  -> keep the projection head and evaluate on its outputs.
#   None  -> use whatever the pretrain config saved (currently True everywhere).
# Overrides whatever the pretrain experiment recorded; logged into eval_config.json.
use_projection = False

# Evaluation hyperparameters. Architecture (embed_dim, num_heads, num_layers,
# patch_size, lambda_factor, dropout, proj_*) is auto-loaded from the pretrain
# experiment — only set those keys here if you deliberately want to deviate.
#
# IMPORTANT: `dataset` must match the dataset this checkpoint was pretrained on.
# A mismatch (e.g. a Trento checkpoint evaluated on "houston") now raises a clear
# band-count error in scripts/evaluate_cosine.py instead of silently scoring a
# randomly-initialized input projection.
eval_params = {
    "dataset": "houston",                   # "houston" | "trento" | "muufl"
    # "n_way": 6,                          # omit to default to all available classes (C-way)
    "k_shot": 5,
    "k_query": 100,
    "num_episodes": 1000,
    "distance_metric": "euclidean",          # "cosine" | "euclidean"
    "temperature": 10.0,
    "prototype_mode": "mean_features",    # "mean_features" | "mean_distances"
    "pool_sigma": None,                    # e.g. 2.0 for center-weighted pooling
    "split": "all",                      # "all" 
    "seed": 42,
    "no_plots": False,
    "num_example_episodes": 10,
    "max_tsne_samples": 100,

    # --- Architecture overrides (rare; defaults come from the pretrain experiment) ---
    # The Trento pretrain config saved lambda_factor=4.0, which is far too high at
    # eval: in adapt_embeddings the per-sample CLS token dominates every patch
    # (z = patch + lambda*cls) and collapses class-discriminative signal. Override
    # to the Houston-proven 0.5. (lambda_factor is unused during pretraining.)
    "lambda_factor": 0.5,
    # "embed_dim": 128, "num_heads": 2, "num_layers": 2,
    # "patch_size": 11, "dropout": 0.1,
}

overwrite = True

In [41]:
# Auto-reload edits in lib/ and scripts/ so we don't need a kernel restart
# every time something in those modules changes.
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Repo root: /work/nmaric/CoFFE/CoFFE


In [58]:
from lib.eval_runner import run_evaluation

# Build the params dict that gets forwarded to the eval script. `device` and
# `use_projection` are surfaced as top-level parameters for clarity; we splice
# them in here so they land in eval_config.json alongside everything else.
forwarded_params = {**eval_params, "device": device}
if use_projection is not None:
    forwarded_params["use_projection"] = use_projection

ev = run_evaluation(
    experiment_name=experiment_name,
    eval_name=eval_name,
    epoch=epoch,
    checkpoint=checkpoint,
    eval_params=forwarded_params,
    overwrite=overwrite,
)
print("Eval dir:", ev.root)

2026-06-12 18:47:46,726 - Loaded architecture from experiments/houston_enhanced_spatial_mask_test_run1_seed52/pretrain_config.yaml: name=mft_cpea, embed_dim=128, num_heads=2, num_layers=2, lambda_factor=0.5, dropout=0.1, use_projection=False, proj_hidden_dim=512, proj_num_layers=1, proj_l2_normalize=True, patch_size=11
2026-06-12 18:47:46,729 - Starting eval 'houston_enhanced_spatial_mask_test_run5_seed52' for experiment 'houston_enhanced_spatial_mask_test_run1_seed52'
2026-06-12 18:47:46,729 - Checkpoint: experiments/houston_enhanced_spatial_mask_test_run1_seed52/checkpoints/checkpoint_epoch_1500.pth
2026-06-12 18:47:46,729 - Using device: cuda:3 (NVIDIA GeForce RTX 4090)
2026-06-12 18:47:48,955 - Loaded houston (all): 15029 samples
2026-06-12 18:47:48,956 - Loading MFT-CPEA-Cosine (distance=euclidean, temp=10.0, mode=mean_features)
2026-06-12 18:47:48,961 - Loading checkpoint from experiments/houston_enhanced_spatial_mask_test_run1_seed52/checkpoints/checkpoint_epoch_1500.pth
2026-06


EVALUATION RESULTS - HOUSTON (15-way 5-shot)
Method: MFT-CPEA-Cosine (distance=euclidean, mode=mean_features)

--------------------------------------------------------------------------------
Class                         Accuracy     ± 95% CI      Samples
--------------------------------------------------------------------------------
Healthy grass                   81.28%        0.89%       100000
Stressed grass                  74.83%        0.97%       100000
Synthetic grass                 96.77%        0.20%       100000
Trees                           87.47%        0.31%       100000
Soil                            96.21%        0.36%       100000
Water                           83.75%        0.29%       100000
Residential                     82.86%        0.47%       100000
Commercial                      47.02%        0.73%       100000
Road                            81.47%        0.64%       100000
Highway                         20.95%        0.86%       100000
Railway    

2026-06-12 18:49:25,267 - All plots saved to /work/nmaric/CoFFE/CoFFE/experiments/houston_enhanced_spatial_mask_test_run1_seed52/evaluations/houston_enhanced_spatial_mask_test_run5_seed52/plots
2026-06-12 18:49:25,271 - Eval 'houston_enhanced_spatial_mask_test_run5_seed52' complete.


Eval dir: /work/nmaric/CoFFE/CoFFE/experiments/houston_enhanced_spatial_mask_test_run1_seed52/evaluations/houston_enhanced_spatial_mask_test_run5_seed52


In [51]:
# Summary metrics from this run.
import json
print(json.dumps(ev.metadata.get("summary", {}), indent=2, default=str))
print("\nArchitecture loaded from pretrain experiment:")
print(json.dumps(ev.config.get("arch_from_pretrain", {}), indent=2))

{
  "OA": {
    "mean": 74.96306666666665,
    "std": 2.2485186076367905,
    "ci_95": 0.13953111535628998
  },
  "AA": {
    "mean": 74.96306666666665,
    "std": 2.24851860763679,
    "ci_95": 0.13953111535628995
  },
  "Kappa": {
    "mean": 73.17471428571429,
    "std": 2.4091270796108466,
    "ci_95": 0.14949762359602495
  },
  "num_episodes": 1000
}

Architecture loaded from pretrain experiment:
{
  "name": "mft_cpea",
  "embed_dim": 128,
  "num_heads": 2,
  "num_layers": 2,
  "lambda_factor": 0.5,
  "dropout": 0.1,
  "use_projection": true,
  "proj_hidden_dim": 512,
  "proj_num_layers": 1,
  "proj_l2_normalize": true,
  "patch_size": 11
}


In [56]:
# Quick look at per-class accuracy from the full results.json.
import json
results = json.loads(ev.results_path.read_text())
per_class = results.get("per_class", {})
for cls, data in sorted(per_class.items(), key=lambda kv: int(kv[0])):
    print(f"class {cls}: {data['accuracy']:6.2f}%  \u00b1 {data['ci_95']:5.2f}%  (n={data['total_samples']})")

class 1:  75.69%  ±  1.43%  (n=100000)
class 2:  44.68%  ±  1.26%  (n=100000)
class 3:  97.12%  ±  0.38%  (n=100000)
class 4:  79.04%  ±  0.61%  (n=100000)
class 5:  98.69%  ±  0.19%  (n=100000)
class 6:  68.13%  ±  0.58%  (n=100000)
class 7:  67.89%  ±  0.65%  (n=100000)
class 8:  20.06%  ±  0.57%  (n=100000)
class 9:  67.42%  ±  1.00%  (n=100000)
class 10:  16.79%  ±  0.84%  (n=100000)
class 11:  33.97%  ±  0.99%  (n=100000)
class 12:  24.62%  ±  0.83%  (n=100000)
class 13:  78.25%  ±  0.69%  (n=100000)
class 14:  98.27%  ±  0.13%  (n=100000)
class 15:  98.86%  ±  0.17%  (n=100000)


In [57]:
import gc, torch


gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()  # optional, releases shared memory handles

# Confirm it actually freed
print(torch.cuda.memory_summary(device=device, abbreviated=True))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 2                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   9344 KiB |   1732 MiB |   5602 GiB |   5602 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   9344 KiB |   1732 MiB |   5602 GiB |   5602 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |   9344 KiB |   1729 MiB |   5592 GiB |   5592 GiB |
|---------------------------------------------------------------